# Ноутбук для решения задачи урока 5.1


In [ ]:
# Импортируем датасет

import pandas as pd

df = pd.read_csv("https://stepik.org/media/attachments/lesson/1028705/mulimodal_questions.csv")
df

In [ ]:
# Установим необходимую версию библиотеки

!pip install bitsandbytes==0.40.0 -qq

In [ ]:
# Скачаем zip-архив с картинками

!wget https://stepik.org/media/attachments/lesson/1028705/images.zip
!unzip images.zip

In [ ]:
from PIL import Image
import torch
from transformers import pipeline
from transformers import BitsAndBytesConfig

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, # подгружаем сразу оптимальную версию
    bnb_4bit_compute_dtype=torch.float16
)

In [ ]:
# указываем тип задачи и модель
model_id = "llava-hf/llava-1.5-7b-hf"

pipe = pipeline("image-to-text", model=model_id, model_kwargs={"quantization_config": quantization_config})

In [ ]:
images = [f"images/im{i}.jpg" for i in range(0, 10)]

In [ ]:
# Получим предсказания для изображений

ans = []
for im_path, question in zip(images, df['question'].values):

    im = Image.open(im_path)

    prompt = f"USER:<image>\n{question}. answer with an int number\nASSISTANT:"
    outputs = pipe(im, prompt=prompt, generate_kwargs={"max_new_tokens": 200})
    outputs = int(outputs[0]['generated_text'].split('ASSISTANT: ')[1])
    ans.append(outputs)
    #break # уберите break, когда убедитесь, что код работает для одного изображения
print(ans)

In [ ]:
# запишем ответы в датафрейм

df['answer'] = ans
df.drop(columns=['image_name'], inplace=True)
df.to_csv('answer.csv', index=False)
df